<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# =========================
# 50_build_app_artefacts.py
# Rebuild app artefacts from model outputs under app-demo/extracted/
# =========================

# I keep everything in one cell so I can just run-and-done in Colab.

import os
import re
import glob
import math
from pathlib import Path
import numpy as np
import pandas as pd

# ---- 0) Drive + paths --------------------------------------------------------

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path("/content/drive/MyDrive/gt-markets")
else:
    # local/dev fallback
    ROOT = Path.home() / "gt-markets"

APP_ARTE = ROOT / "AppDemo" / "artefacts"
APP_ARTE.mkdir(parents=True, exist_ok=True)

# canonical input folders
SRC_EXTRACT_D = ROOT / "app-demo" / "extracted" / "daily"
SRC_EXTRACT_W = ROOT / "app-demo" / "extracted" / "weekly"

print("Mounted drive, artefacts dir:", APP_ARTE)

# ---- 1) Helper utils ---------------------------------------------------------

def _warn(msg):
    print(f"[warn] {msg}")

def _info(msg):
    print(f"[info] {msg}")

def read_csv_safe(p: Path):
    try:
        return pd.read_csv(p)
    except Exception as e:
        _warn(f"failed to read {p.name}: {e}")
        return None

def latest_engineered_trends_csv(root: Path) -> Path | None:
    # I grab the newest merged_financial_trends_engineered_*.csv if available (used to backfill Close)
    candidates = sorted(root.glob("merged_financial_trends_engineered_*.csv"))
    return candidates[-1] if candidates else None

# I map asset -> engineered Close column (from trends CSV) so I can backfill if model file lacks Close
ENGINEERED_CLOSE_MAP = {
    "GOLD":   "GC=F Close",
    "OIL":    "CL=F Close",
    "BTC":    "BTC-USD Close",
    "USDCNY": "USDCNY=X Close",
}

ASSET_FROM_PREFIX = {
    "gold": "GOLD",
    "oil": "OIL",
    "btc": "BTC",
    "usdcny": "USDCNY",
}

# model parser: I only need the model tag from filenames like:
# gold_eng_ext_lstm_w30.val_f0.csv, btc_raw_base_rf.test_f0.csv, etc.
MODEL_TAG_RE = re.compile(
    r"^(?P<prefix>gold|oil|btc|usdcny)_[^_]+_[^_]+_(?P<model>[^.]+)\.(?P<split>val|test)_f\d+\.csv$",
    re.IGNORECASE
)

def parse_file_meta(fname: str):
    m = MODEL_TAG_RE.match(fname)
    if not m:
        return None
    prefix = m.group("prefix").lower()
    asset = ASSET_FROM_PREFIX.get(prefix)
    model = m.group("model")  # e.g., lr, rf, xgb, mlp_w30, lstm_w30, gru_w30
    split = m.group("split")  # val | test (I don't filter by it; I just treat both equally)
    return asset, model, split

# metrics calc (I keep it lightweight and robust)
def compute_metrics(df: pd.DataFrame, freq: str) -> dict:
    # Expect df with Date, Close, signal (long-only 0/1 or -1/0/1). I apply 1-day lag on signal.
    df = df.sort_values("Date").copy()
    df["ret"] = df["Close"].pct_change().fillna(0.0)

    sig = df["signal"].fillna(0.0).astype(float)
    # I interpret any negative as -1 (short), any positive as +1 (long), else 0 (flat)
    sig = sig.where(sig == 0, np.sign(sig))
    sig_shift = sig.shift(1).fillna(0.0)

    port_ret = sig_shift * df["ret"]

    if freq.upper() == "D":
        n_per_year = 252
    else:
        n_per_year = 52

    avg = port_ret.mean()
    vol = port_ret.std(ddof=1)
    sharpe = (avg / vol) * math.sqrt(n_per_year) if vol > 0 else 0.0

    # CAGR (I protect against negatives by compounding (1+r); if any <= -1, cap to small epsilon)
    gross = (1.0 + port_ret.clip(lower=-0.9999)).prod()
    n = max(len(port_ret), 1)
    cagr = gross**(n_per_year / n) - 1.0

    # hit-rate
    win_rate = (port_ret > 0).mean()

    # max drawdown on equity curve
    eq = (1.0 + port_ret).cumprod()
    roll_max = eq.cummax()
    dd = (eq / roll_max) - 1.0
    max_dd = dd.min()

    return {
        "Return_Ann": float(cagr),
        "Sharpe": float(sharpe),
        "WinRate": float(win_rate),
        "MaxDD": float(max_dd),
        "N_Trades": int((sig_shift.diff().abs() > 0).sum()),
    }

# make sure per-asset folders exist
for a in ["GOLD", "BTC", "OIL", "USDCNY"]:
    (APP_ARTE / a).mkdir(parents=True, exist_ok=True)

# ---- 2) Load engineered Close (optional) -------------------------------------

ENGINEERED_PATH = latest_engineered_trends_csv(ROOT)
engineered_df = None
if ENGINEERED_PATH and ENGINEERED_PATH.exists():
    _info(f"found engineered trends: {ENGINEERED_PATH.name}")
    engineered_df = pd.read_csv(ENGINEERED_PATH, parse_dates=["Date"])
    engineered_df = engineered_df[["Date"] + list(ENGINEERED_CLOSE_MAP.values())].drop_duplicates("Date")
else:
    _warn("no engineered trends file found -> will rely on Close in extracted files only")

def backfill_close(df: pd.DataFrame, asset: str) -> pd.DataFrame:
    if "Close" in df.columns:
        return df
    if engineered_df is None:
        _warn("engineered trends not loaded; can't backfill Close")
        return df
    col = ENGINEERED_CLOSE_MAP.get(asset)
    if not col or col not in engineered_df.columns:
        _warn(f"no Close map for {asset}; can't backfill")
        return df
    # merge on Date
    merged = df.merge(engineered_df[["Date", col]], on="Date", how="left")
    if col in merged:
        merged["Close"] = merged[col]
        merged = merged.drop(columns=[col])
        _info(f"mapped Close for {asset}: {col}")
    if "Close" not in merged.columns:
        _warn(f"still no Close after merge for {asset}")
    return merged

# ---- 3) Scan extracted model outputs ----------------------------------------

def collect_model_rows(src_folder: Path, freq: str) -> list[dict]:
    rows = []
    if not src_folder.exists():
        _warn(f"missing extracted folder: {src_folder}")
        return rows

    for p in sorted(src_folder.glob("*.csv")):
        meta = parse_file_meta(p.name)
        if not meta:
            # ignore non-conforming files
            continue
        asset, model_tag, split = meta

        df = read_csv_safe(p)
        if df is None or df.empty:
            continue

        # normalize schema
        # I try to find signals in the usual suspects:
        #  - 'position' (already -1/0/+1 or 0/1)
        #  - 'signal'
        #  - else derive from 'prob_up' (long if > 0.5, flat otherwise)
        # also normalize Date and Close
        # Accept 'date' or 'Date' and coerce to datetime
        date_col = "Date" if "Date" in df.columns else ("date" if "date" in df.columns else None)
        if not date_col:
            _warn(f"no Date column in {p.name}; skipped")
            continue

        df = df.rename(columns={date_col: "Date"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

        # Close backfill if needed
        if "Close" not in df.columns or df["Close"].isna().all():
            df = backfill_close(df, asset)
            if "Close" not in df.columns or df["Close"].isna().all():
                _warn(f"missing Close in {p.name}; skipped")
                continue

        # build a clean signal
        signal = None
        if "position" in df.columns:
            signal = df["position"].astype(float)
            strategy_rule = "ModelPosition"    # uses model-produced position
        elif "signal" in df.columns:
            signal = df["signal"].astype(float)
            strategy_rule = "ModelSignal"      # uses model-produced signal
        elif "prob_up" in df.columns:
            # long if prob_up > 0.5, else flat (this is our KeywordsRule_v1 for models without explicit positions)
            signal = (df["prob_up"].astype(float) > 0.5).astype(int)
            strategy_rule = "KeywordsRule_v1"
        else:
            _warn(f"no position/signal/prob_up in {p.name}; skipped")
            continue

        tmp = pd.DataFrame({"Date": df["Date"], "Close": df["Close"], "signal": signal})
        tmp = tmp.dropna(subset=["Close"]).reset_index(drop=True)
        if tmp.empty:
            _warn(f"no usable rows after cleaning in {p.name}; skipped")
            continue

        # compute metrics
        m = compute_metrics(tmp, freq=freq)

        # I keep training model separate from trading rule to avoid confusion
        rows.append({
            "type": "model",           # for app grouping (baseline vs model)
            "asset": asset,
            "freq": freq,
            "strategy": strategy_rule, # trading rule
            "model": model_tag,        # training model
            "split": split,            # val/test (FYI)
            **m
        })

    return rows

rows_D = collect_model_rows(SRC_EXTRACT_D, "D")
rows_W = collect_model_rows(SRC_EXTRACT_W, "W")

# Export per-asset model metrics for the app (keywords_* files)
def export_keywords_metrics(rows: list[dict], freq: str):
    if not rows:
        _warn(f"no model metrics for {freq}")
        # still write empty files so app shows a warning instead of crashing
        for a in ["GOLD", "BTC", "OIL", "USDCNY"]:
            out = APP_ARTE / a / f"metrics_keywords_{freq}.csv"
            pd.DataFrame(columns=["type","asset","freq","strategy","model","split",
                                  "Return_Ann","Sharpe","WinRate","MaxDD","N_Trades"]).to_csv(out, index=False)
        return

    df = pd.DataFrame(rows)
    for a in df["asset"].unique():
        out = APP_ARTE / a / f"metrics_keywords_{freq}.csv"
        df[df["asset"] == a].sort_values(["asset","strategy","model"]).to_csv(out, index=False)

export_keywords_metrics(rows_D, "D")
export_keywords_metrics(rows_W, "W")
print("✅ Exported model metrics (keywords)")

# ---- 4) Merge with baseline metrics if present -------------------------------

def try_read_baseline(asset: str, freq: str) -> pd.DataFrame:
    p = APP_ARTE / asset / f"metrics_baseline_{freq}.csv"
    if not p.exists():
        _warn(f"baseline not found: {p}")
        return pd.DataFrame()
    df = pd.read_csv(p)
    # normalize columns for the app summary (baseline has no 'model'/'split')
    df["type"] = "baseline"
    df["asset"] = asset
    df["freq"] = freq
    if "strategy" not in df.columns:
        # I align to our expected schema
        if "rule" in df.columns:
            df = df.rename(columns={"rule": "strategy"})
        else:
            df["strategy"] = "baseline_rule"
    for c in ["Return_Ann","Sharpe","WinRate","MaxDD","N_Trades"]:
        if c not in df.columns:
            df[c] = np.nan
    df["model"] = ""
    df["split"] = ""
    cols = ["type","asset","freq","strategy","model","split","Return_Ann","Sharpe","WinRate","MaxDD","N_Trades"]
    return df[cols]

def read_keywords(freq: str) -> pd.DataFrame:
    frames = []
    for a in ["GOLD","BTC","OIL","USDCNY"]:
        p = APP_ARTE / a / f"metrics_keywords_{freq}.csv"
        if p.exists():
            frames.append(pd.read_csv(p))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def build_summary(freq: str):
    base_frames = [try_read_baseline(a, freq) for a in ["GOLD","BTC","OIL","USDCNY"]]
    baseline_df = pd.concat([f for f in base_frames if not f.empty], ignore_index=True) if any([not f.empty for f in base_frames]) else pd.DataFrame()

    keywords_df = read_keywords(freq)
    cols = ["type","asset","freq","strategy","model","split","Return_Ann","Sharpe","WinRate","MaxDD","N_Trades"]

    if not baseline_df.empty and not keywords_df.empty:
        out = pd.concat([baseline_df[cols], keywords_df[cols]], ignore_index=True)
    elif not baseline_df.empty:
        out = baseline_df[cols]
    else:
        out = keywords_df[cols]

    out = out.sort_values(["asset","type","strategy","model"]).reset_index(drop=True)
    out_path = APP_ARTE / f"metrics_summary_{freq}.csv"
    out.to_csv(out_path, index=False)
    return out

summary_D = build_summary("D")
summary_W = build_summary("W")

print("\n✅ Exported combined summary metrics (D + W)\n")

print("[Sample summary_D]")
print(summary_D.head(10) if not summary_D.empty else summary_D)

print("\n[Sample summary_W]")
print(summary_W.head(10) if not summary_W.empty else summary_W)

# ---- 5) Quick app preflight (optional) ---------------------------------------

_required = {
    "GOLD": ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "BTC":  ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "OIL":  ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "USDCNY":["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
}

missing = []
for asset, files in _required.items():
    for f in files:
        if not (APP_ARTE/asset/f).exists():
            missing.append(f"{asset}/{f}")

if missing:
    _warn("Missing artefact files:\n  - " + "\n  - ".join(missing))
else:
    print("✅ All required artefact files present.")

Mounted at /content/drive
Mounted drive, artefacts dir: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
[warn] no engineered trends file found -> will rely on Close in extracted files only
[warn] no model metrics for D
[warn] no model metrics for W
✅ Exported model metrics (keywords)

✅ Exported combined summary metrics (D + W)

[Sample summary_D]
       type   asset freq  strategy model split  Return_Ann    Sharpe  WinRate  \
0  baseline     BTC    D  BASE_EMA                0.667163  1.300460      NaN   
1  baseline     BTC    D  BASE_SMA                0.703561  1.298150      NaN   
2  baseline    GOLD    D  BASE_EMA                0.073589  0.609866      NaN   
3  baseline    GOLD    D  BASE_SMA                0.099425  0.790527      NaN   
4  baseline     OIL    D  BASE_EMA                0.065588  0.263851      NaN   
5  baseline     OIL    D  BASE_SMA                0.034963  0.145772      NaN   
6  baseline  USDCNY    D  BASE_EMA                0.016960  0.456281      NaN  